In [11]:
# 02 - Data Cleaning Pipeline
## Cell 1: Environment Setup & Raw HR Table Ingestion

import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw")
employees = pd.read_csv(RAW / 'employees.csv')
compensation = pd.read_csv(RAW / 'compensation.csv')
performance = pd.read_csv(RAW / 'performance_reviews.csv')
exit_surveys = pd.read_csv(RAW / 'exit_surveys.csv')
departments = pd.read_csv(RAW / 'departments.csv')

print(f"Starting employee rows: {len(employees):,}")


Starting employee rows: 3,347


In [12]:
## Cell 2: Fix 1 - Drop Duplicate employee_id Rows

before = len(employees)
employees = employees.drop_duplicates(subset=['employee_id'], keep='first')
print(f"Removed {before - len(employees)} duplicates. Remaining: {len(employees):,}")


Removed 47 duplicates. Remaining: 3,300


In [13]:
## Cell 3: Fix 2 - Standardize Department Labels (27 Raw Variants -> 7 Canonical Values)

CANONICAL = {d.strip().upper(): d for d in departments['department_name']}

def clean_dept(d):
    return CANONICAL.get(str(d).strip().upper(), str(d).strip())

before_n = employees['department'].nunique()
employees['department'] = employees['department'].apply(clean_dept)
print(f"Department labels: {before_n} raw variants -> {employees['department'].nunique()} canonical values")
print(employees['department'].value_counts())


Department labels: 27 raw variants -> 7 canonical values
department
Engineering         1165
Sales                669
Customer Success     427
Marketing            348
Finance              299
Operations           196
HR                   196
Name: count, dtype: int64


In [14]:
## Cell 4: Fix 3 - Flag the Status-Sync Gap Instead of Guessing a Termination Date

employees['hire_date'] = pd.to_datetime(employees['hire_date'], errors='coerce')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce')

gap = (employees['active_status'] == 0) & (employees['termination_date'].isna())
employees['data_quality_flag'] = None
employees.loc[gap, 'data_quality_flag'] = 'MISSING_TERM_DATE'

print(f"{gap.sum()} rows flagged MISSING_TERM_DATE (not imputed — a guessed exit date would corrupt tenure analysis).")


122 rows flagged MISSING_TERM_DATE (not imputed — a guessed exit date would corrupt tenure analysis).


In [15]:
## Cell 5: Fix 4 - Correct Swapped Hire and Termination Dates

swapped = employees['termination_date'].notna() & (employees['hire_date'] > employees['termination_date'])
print(f"{swapped.sum()} rows had swapped dates — correcting.")

tmp = employees.loc[swapped, 'hire_date'].copy()
employees.loc[swapped, 'hire_date'] = employees.loc[swapped, 'termination_date']
employees.loc[swapped, 'termination_date'] = tmp

print(f"Remaining swapped dates: {(employees['hire_date'] > employees['termination_date']).sum()}")


33 rows had swapped dates — correcting.
Remaining swapped dates: 0


In [16]:
## Cell 6: Fix 5 & 6 - Flag Invalid Distance and Invalid Salary (Don't Impute)

employees['is_valid_distance'] = employees['distance_from_home_miles'] > 0
compensation['is_valid_salary'] = compensation['base_salary'] > 0

print(f"{(~employees['is_valid_distance']).sum()} invalid distance rows flagged.")
print(f"{(~compensation['is_valid_salary']).sum()} invalid salary rows flagged.")


58 invalid distance rows flagged.
61 invalid salary rows flagged.


In [17]:
## Cell 7: Fix 7 - Flag Invalid Performance Scores (Outside 1-5 Scale)

performance['is_valid_score'] = performance['performance_score'].between(1, 5)
print(f"{(~performance['is_valid_score']).sum()} invalid performance scores flagged.")


41 invalid performance scores flagged.


In [18]:
## Cell 8: Fix 8 - Drop Orphaned Exit-Survey Rows

before_exit = len(exit_surveys)
exit_surveys = exit_surveys[exit_surveys['employee_id'].isin(employees['employee_id'])]
print(f"Dropped {before_exit - len(exit_surveys)} orphaned exit survey rows. Remaining: {len(exit_surveys):,}")


Dropped 24 orphaned exit survey rows. Remaining: 967


In [19]:
## Cell 9: Save Cleaned Tables to data/processed/

OUT = Path(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\processed")
OUT.mkdir(exist_ok=True)

employees.to_csv(OUT / 'employees_clean.csv', index=False)
compensation.to_csv(OUT / 'compensation_clean.csv', index=False)
performance.to_csv(OUT / 'performance_reviews_clean.csv', index=False)
exit_surveys.to_csv(OUT / 'exit_surveys_clean.csv', index=False)
departments.to_csv(OUT / 'departments_clean.csv', index=False)

print(f"Final employees table: {len(employees):,} rows - All 5 cleaned tables saved!")


Final employees table: 3,300 rows - All 5 cleaned tables saved!
